In [1]:
%%duckdb


SELECT 
  *, date_diff('minute', start_at, stop_at) AS duration_min
FROM (
  SELECT 
    * EXCLUDE (tripduration, starttime, stoptime),
    strptime(starttime, ['%m/%d/%Y %H:%M', '%m/%d/%Y %H:%M:%S', '%Y-%m-%d %H:%M:%S']) AS start_at,
    strptime(stoptime, ['%m/%d/%Y %H:%M', '%m/%d/%Y %H:%M:%S', '%Y-%m-%d %H:%M:%S']) AS stop_at
  FROM 's3://tt-bootcamp-shared/nyc-bike-trip/201[56]*.parquet'
)
LIMIT 3

,start station id,start station name,start station latitude,start station longitude,end station id,end station name,end station latitude,end station longitude,bikeid,usertype,birth year,gender,start_at,stop_at,duration_min
0,455,1 Ave & E 44 St,40.750020,-73.969053,265,Stanton St & Chrystie St,40.722293,-73.991475,18660,Subscriber,1960.0,2,2015-01-01 00:01:00,2015-01-01 00:24:00,23
1,434,9 Ave & W 18 St,40.743174,-74.003664,482,W 15 St & 7 Ave,40.739355,-73.999318,16085,Subscriber,1963.0,1,2015-01-01 00:02:00,2015-01-01 00:08:00,6
2,491,E 24 St & Park Ave S,40.740964,-73.986022,505,6 Ave & W 33 St,40.749013,-73.988484,20845,Subscriber,1974.0,1,2015-01-01 00:04:00,2015-01-01 00:10:00,6


## Problem 1: Trip duration

### Part 1: Build a Regression Model

Build a regression to predict trip duration by using
- Day of time
- Distance between start and end stations (there might be more than one way to measure it)
- Hour of day
- Weekend indicator
- Don't forget to model bias (this one is intentionally not used in lecture)
- Also any thing you want to end

### Part 2: Experiment Design

- Ensure that you properly design your experiment to report unbiased performance metric you choose

### Part 3 [Optional]: Visualize

- Generate some fictional pickup and dropoff locations for bike trips (random pair selection)
- Estimate trip duration for those say 10 trips
- Visualize them on map using `pydeck` by using redish color for slower trips and greener for faster trips.

## -- ÇÖZÜM --

#### -> DuckDB SQL sorgusu ile öznitelik mühendisliği yapılarak gereken öznitelikler çıkarıldı ve bellekte taşma yapmaması için view olarak create edildi. 
#### -> Mesafe hesaplarken, gerçek hayat sürüş mesafesine uygunluğu sebebiyle Manhattan Uzaklık formülü kullanıldı.
#### -> Aykırı konum değeri olmaması için New York lat long değerleri ile veri filtrelenmiştir.
#### -> Eğitimin ilerleyen aşamalarında RMSE skorunun MAE'ye kıyasla aşırı yüksek gelmesi sebebiyle outlier dengelemesi yapmak için sorguya (WHERE duration_min BETWEEN 0.1 AND 200.0;) eklendi. 

In [2]:
%%duckdb

CREATE OR REPLACE VIEW bike_trips_features AS
WITH data AS (
  SELECT *, 
    strptime(starttime, ['%m/%d/%Y %H:%M', '%m/%d/%Y %H:%M:%S', '%Y-%m-%d %H:%M:%S']) AS start_at,
    strptime(stoptime, ['%m/%d/%Y %H:%M', '%m/%d/%Y %H:%M:%S', '%Y-%m-%d %H:%M:%S']) AS stop_at
  FROM 's3://tt-bootcamp-shared/nyc-bike-trip/201[56]*.parquet'
),
processed AS (
  SELECT date_diff('second', start_at, stop_at) / 60.0 AS duration_min,
    "start station latitude" AS ss_lat,
    "start station longitude" AS ss_lon,
    "end station latitude" AS es_lat,
    "end station longitude" AS es_lon,
    sqrt(pow(("start station latitude" - "end station latitude") * 111.0, 2) + 
     pow(("start station longitude" - "end station longitude") * 84.3, 2)) AS euc_km,
    (abs("start station latitude" - "end station latitude") * 111.0 + 
     abs("start station longitude" - "end station longitude") * 84.3) AS block_km,
    hour(start_at) AS hour_of_day,
    dayofweek(start_at) AS day_of_week,
    case when dayofweek(start_at) IN (6, 7) THEN 1 ELSE 0 END AS is_weekend
  FROM data
  WHERE 
    "start station latitude" >  40.730610 AND "start station longitude" < -73.935242
    AND "end station latitude" > 40.730610 AND "end station longitude" < -73.935242
)
SELECT * 
FROM processed
WHERE duration_min BETWEEN 0.1 AND 200.0;

,Count


In [3]:
%%duckdb -o df

SELECT * FROM bike_trips_features;

,duration_min,ss_lat,ss_lon,es_lat,es_lon,euc_km,block_km,hour_of_day,day_of_week,is_weekend
0,6.000000,40.743174,-74.003664,40.739355,-73.999318,0.560328,0.790335,0,4,0
1,6.000000,40.740964,-73.986022,40.749013,-73.988484,0.917222,1.100967,0,4,0
2,8.000000,40.750073,-73.998393,40.735238,-74.000271,1.654246,1.804985,0,4,0
3,2.000000,40.748549,-73.988084,40.745168,-73.986831,0.389873,0.480943,0,4,0
4,20.000000,40.739323,-74.008119,40.738177,-73.977387,2.593863,2.718001,0,4,0
...,...,...,...,...,...,...,...,...,...,...
11711395,20.983333,40.742388,-73.997262,40.756458,-73.993722,1.590068,1.860244,23,6,1
11711396,2.033333,40.758985,-73.993800,40.760301,-73.998842,0.449489,0.571178,23,6,1
11711397,17.116667,40.764397,-73.973715,40.776829,-73.963888,1.609470,2.208313,23,6,1
11711398,29.133333,40.791976,-73.945993,40.791270,-73.964839,1.590649,1.667084,23,6,1


In [4]:
df.head(3)

,duration_min,ss_lat,ss_lon,es_lat,es_lon,euc_km,block_km,hour_of_day,day_of_week,is_weekend
0,6.0,40.743174,-74.003664,40.739355,-73.999318,0.560328,0.790335,0,4,0
1,6.0,40.740964,-73.986022,40.749013,-73.988484,0.917222,1.100967,0,4,0
2,8.0,40.750073,-73.998393,40.735238,-74.000271,1.654246,1.804985,0,4,0


### ----------
#### -> Sorgu sonucu elde edilen veri, dışarı aktarıldı ve scikitlearn kütüphanesinde bulunan hazır LinearRegression algoritmasına oturtuldu. (! Burada test yapılmadı sadece bias gözlemlendi !)


In [ ]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression

X = df[['block_km', 'euc_km', 'hour_of_day', 'day_of_week', 'is_weekend']].copy()
X = pd.get_dummies(X, columns=['hour_of_day', 'day_of_week'], drop_first=True)
y = df['duration_min']

model = LinearRegression(fit_intercept=True)
model.fit(X, y)

print(f"Bias -> {model.intercept_:.2f} (minute type)")

### ----------
#### -> 085_Func.ipynb dersinde saf python ile kodlamasını öğrendiğimiz değerlendirme metrikleri scikit-learn ile import edildi
#### -> Veri seti train_test_split ile ayrılarak veri sızıntısı engellendi
#### -> Harici test verisi ile yapılan değerlendirme sonucu elde edilen mae, rmse değerlendirme metrikleri ve bias değeri gözlemlendi

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, root_mean_squared_error, r2_score

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

experimental_designed_model = LinearRegression(fit_intercept=True)
experimental_designed_model.fit(X_train, y_train)

y_test_pred = experimental_designed_model.predict(X_test)

test_mae = mean_absolute_error(y_test, y_test_pred)
test_rmse = root_mean_squared_error(y_test, y_test_pred)
test_r2 = r2_score(y_test, y_test_pred)

print(f"External Test MAE -> {test_mae:.2f} dk")
print(f"External Test RMSE -> {test_rmse:.2f} dk")
print(f"External Test R^2 -> {test_r2:.2f}")
print(f"Bias ->: {experimental_designed_model.intercept_:.2f} (minute type)")

### Problem 2: Extending Naive Bayesian

### Part 1: Expand the NB Regression Idea to continous variable

$$
P(gender = a, speed_{bike} = x) = P(gender = a) P(speed_{bike} = x | gender = a)
$$

- Note that $P(speed_{bike} = x | gender = a)$ is  continous distribution.
- Expand the idea
- Build a predictive model for estimation biker gender using the bike speed ?

### Part 2: Use Visualization to decide best distribution 

- How should be $P(speed_{bike} = x | gender = a)$ modeled

### -- ÇÖZÜM --
#### -> Gerçek hayata uygunluğu sebebiyle Manhattan Uzaklık formülü ile distance hesaplanmıştır.
#### -> speed km/h cinsinden hesaplanmıştır.
#### -> Aykırı konum değeri olmaması için New York lat long değerleri ile veri filtrelenmiştir.
#### -> Aykırı hız değeri olmaması için veri heuristic olarak 50 km/h ile filtrelenmiştir.

In [10]:
%%duckdb -o df_speed_by_gender

CREATE OR REPLACE VIEW speed_by_gender AS
WITH speed_by_gender_data AS (
  SELECT *, 
    strptime(starttime, ['%m/%d/%Y %H:%M', '%m/%d/%Y %H:%M:%S', '%Y-%m-%d %H:%M:%S']) AS start_at,
    strptime(stoptime, ['%m/%d/%Y %H:%M', '%m/%d/%Y %H:%M:%S', '%Y-%m-%d %H:%M:%S']) AS stop_at
  FROM 's3://tt-bootcamp-shared/nyc-bike-trip/201[56]*.parquet'),
data AS (
  SELECT gender, date_diff('second', start_at, stop_at) / 60.0 AS duration_min,
    (abs("start station latitude" - "end station latitude") * 111.0 + 
     abs("start station longitude" - "end station longitude") * 84.3) AS block_km
  FROM speed_by_gender_data
  WHERE 
    "start station latitude" > 40.730610 AND "start station longitude" < -73.935242
    AND "end station latitude" > 40.730610 AND "end station longitude" < -73.935242
    AND gender IN (1, 2))
SELECT gender,duration_min, block_km,
  block_km / (duration_min / 60.0) AS speed_kmh
FROM data
WHERE duration_min BETWEEN 2.0 AND 60.0 AND block_km > 0.1;

SELECT * FROM speed_by_gender
WHERE speed_kmh<50;

,gender,duration_min,block_km,speed_kmh
0,1,6.000000,0.790335,7.903352
1,1,6.000000,1.100967,11.009671
2,2,8.000000,1.804985,13.537386
3,1,2.000000,0.480943,14.428287
4,2,20.000000,2.718001,8.154003
...,...,...,...,...
10117004,1,20.983333,1.860244,5.319205
10117005,1,2.033333,0.571178,16.854426
10117006,1,17.116667,2.208313,7.740922
10117007,2,29.133333,1.667084,3.433353


### ----------
#### Dağılım grafiği incelendiğinde erkeklerin hızı, kadınların hızından az da olsa fazla olma eğilimindedir bulgusuna ulaşabilmekteyiz.
#### İki cinsiyet için de yoğunluk/hız grafiği normale yakın dağılmıştır.

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
import numpy as np
import pandas as pd

def normal_probabilitydf(x, mean, std):
    return (1.0 / (std * np.sqrt(2 * np.pi))) * np.exp(-((x - mean) ** 2) / (2 * std ** 2))

def lognormal_probabilitydf(x, mean_ln, std_ln):
    probabilitydf = np.zeros_like(x, dtype=float)
    mask = x > 0
    probabilitydf[mask] = (1.0 / (x[mask] * std_ln * np.sqrt(2 * np.pi))) * np.exp(-((np.log(x[mask]) - mean_ln) ** 2) / (2 * std_ln ** 2))
    return probabilitydf

df_m = df_speed_by_gender[df_speed_by_gender['gender'] == 1]['speed_kmh'].values
df_f = df_speed_by_gender[df_speed_by_gender['gender'] == 2]['speed_kmh'].values

mu_m, std_m = np.mean(df_m), np.std(df_m)
mu_f, std_f = np.mean(df_f), np.std(df_f)

mean_ln_m, std_ln_m = np.mean(np.log(df_m)), np.std(np.log(df_m))
mean_ln_f, std_ln_f = np.mean(np.log(df_f)), np.std(np.log(df_f))

df_speed_by_gender['Cinsiyet'] = df_speed_by_gender['gender'].map({1: 'Male', 2: 'Female'})

fig = px.histogram(
    df_speed_by_gender, x="speed_kmh", color="Cinsiyet", histnorm="probability density", 
    barmode="overlay", nbins=50, title="Speed Distribution by Gender",
)

x_vals = np.linspace(0.1, 35, 500)

fig.add_trace(go.Scatter(x=x_vals, y=normal_probabilitydf(x_vals, mu_m, std_m), name="Male Normal Fit", line=dict(color="blue", dash="dash")))
fig.add_trace(go.Scatter(x=x_vals, y=lognormal_probabilitydf(x_vals, mean_ln_m, std_ln_m), name="Male Log-Normal Fit", line=dict(color="blue")))

fig.add_trace(go.Scatter(x=x_vals, y=normal_probabilitydf(x_vals, mu_f, std_f), name="Female Normal Fit", line=dict(color="red", dash="dash")))
fig.add_trace(go.Scatter(x=x_vals, y=lognormal_probabilitydf(x_vals, mean_ln_f, std_ln_f), name="Female Log-Normal Fit", line=dict(color="red")))

fig.update_layout(xaxis_title="Speed (km/h)", yaxis_title="Density", template="plotly_white")
fig.show()

### ----------
#### Veri seti %80 test, %20 eğitim veri seti olarak ayrılmıştır.
#### Prior Probabilityler bulunmuştur.
#### Ardından Prior Probabilityler kullanılarak Posterior Probabilityler bulunmuştur.
#### Naive Bayes algoritmasında sıfıra gitme problemi bulunduğu için veri logaritma ile ölçeklenmiştir.

In [ ]:
np.random.seed(42)
msk = np.random.rand(len(df_speed_by_gender)) < 0.8
train_df = df_speed_by_gender[msk]
test_df = df_speed_by_gender[~msk]

# Prior Probabilites
n_train = len(train_df)
p_m = len(train_df[train_df['gender'] == 1]) / n_train
p_f = len(train_df[train_df['gender'] == 2]) / n_train
print(f"Priors -> Erkek: {p_m:.4f} | Kadın: {p_f:.4f}\n")

train_m = train_df[train_df['gender'] == 1]['speed_kmh'].values
train_f = train_df[train_df['gender'] == 2]['speed_kmh'].values

mean_ln_m, std_ln_m = np.mean(np.log(train_m)), np.std(np.log(train_m))
mean_ln_f, std_ln_f = np.mean(np.log(train_f)), np.std(np.log(train_f))

from math import log, exp
from sklearn.metrics import classification_report

def predict_gender(speed, return_prob:bool = False):
    log_speed = log(speed)
    
    log_p_m = log(p_m) - log(std_ln_m) - ((log_speed - mean_ln_m) ** 2) / (2 * std_ln_m ** 2)
    log_p_f = log(p_f) - log(std_ln_f) - ((log_speed - mean_ln_f) ** 2) / (2 * std_ln_f ** 2)

    # PROBABILITIES TRANSFORMED LOG TO EXP CAUSE OF PREVENT CONVERGING TO ZERO RISK
    p_m_speed = exp(log_p_m)
    p_f_speed = exp(log_p_f)
    prob_m = p_m_speed / (p_m_speed + p_f_speed)
    prob_f = p_f_speed / (p_m_speed + p_f_speed)

    prediction = 1 if prob_m > prob_f else 2
    
    if return_prob:
        return prediction, prob_m, prob_f
    return prediction

y_true = test_df['gender'].values
y_pred = [predict_gender(s) for s in test_df['speed_kmh'].values]



for idx, sample in test_df.head(5).iterrows():
    speed = sample['speed_kmh']
    true_gender = "Male" if sample['gender'] == 1 else "Female"
    pred, prob_m, prob_f = predict_gender(speed, return_prob=True)
    pred_gender = "Male" if pred == 1 else "Female"
    print(f"Probability of Male = {prob_m:.2%}  - Probability of Female = {prob_f:.2%} - True Label: {true_gender} - Prediction: {pred_gender}")

print("\n")
print(classification_report(y_true, y_pred, target_names=['Male', 'Female']))